In [22]:
import pandas as pd
import glob
from datetime import date, timedelta
import numpy as np
from datetime import datetime
import pathlib
from pathlib import Path
from collections import OrderedDict
import polars as pl
import fastexcel
import os
import time

In [23]:
def convert_to_datetime(struct_time):
    return datetime(*struct_time[:6])

def input_data(folder_path, sheet_name=None, filename_keyword=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")

    if filename_keyword:
        kw = filename_keyword.lower()
        file_paths = [f for f in file_paths if kw in os.path.basename(f).lower()]

    df_list = []
    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name, engine='calamine')
            df = df.select(pl.all().cast(pl.String))
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8", infer_schema_length=0, ignore_errors=True)
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=0)

        # Normalize column names: strip whitespace + BOM
        df.columns = [c.strip().replace('\ufeff', '').replace('\u200b', '') for c in df.columns]

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)

    if df_list:
        return pl.concat(df_list, how='diagonal_relaxed')
    return pl.DataFrame()


def input_data_parquet(folder_path):
    file_paths = glob.glob(f"{folder_path}/*.parquet")
    df_list = []
    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))
        df = pl.read_parquet(file).with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)
    if df_list:
        return pl.concat(df_list, how='vertical')
    return pl.DataFrame()


today_temp = datetime.today().date()
today = today_temp.strftime('%b_%d_%Y')

def pre_process_fcr_excel(folder_path: str):
    excel_files = glob.glob(os.path.join(folder_path, "*.xlsx")) + glob.glob(os.path.join(folder_path, "*.xls"))
    for file_path in excel_files:
        try:
            df = pl.read_excel(file_path, engine="calamine", has_header=False)
            if df.height < 2:
                continue
            header_row_index = 0
            for i in range(min(10, df.height)):
                row_data = [str(x).strip() for x in df.row(i) if x is not None]
                if "FCR Category" in row_data or "Conversation Id" in row_data:
                    header_row_index = i
                    break
            raw_headers = df.row(header_row_index)
            headers = []
            seen = set()
            for i, c in enumerate(raw_headers):
                col_name = str(c).strip() if c is not None and str(c).strip() != "" else f"col_{i}"
                if col_name in seen:
                    col_name = f"{col_name}_dup_{i}"
                seen.add(col_name)
                headers.append(col_name)
            df = df.slice(header_row_index + 1)
            df.columns = headers
            df.write_csv(os.path.splitext(file_path)[0] + ".csv")
        except Exception:
            pass

In [24]:
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Not found the path: {test_path}")

folder_paths = {
    "input_performance":     f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data',
    "output_miv_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/MIV/MIV_Data',
    "hc_extend_by_month":    f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
    "input_survey":          f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN',
    "input_afcr":            f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR',
    "input_t3":              f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN',
    "input_iex":             f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX',
    "mapping_file":          f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx',
    "global_hc":             f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/Global_HC.parquet',
    "input_csv_re_direct":   f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/RE-DIRECT',
    "input_delayed_closure": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE',
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")
print("-" * 60)

pre_process_fcr_excel(folder_paths["input_afcr"])

IEX = input_data(folder_paths["input_iex"]).unique()
IEX = IEX.with_columns([
    pl.col(['Date']).str.to_date("%Y-%m-%d", strict=False),
    pl.col(['Datetime_Fluctuate_Start_Shift','Datetime_Fluctuate_End_Shift',
            'Datetime_First_Start_Shift','Datetime_First_End_Shift'])
      .str.to_datetime("%Y-%m-%d %H:%M:%S.%f", strict=False),
    pl.col(['Night_Shift','Target','Unplanned','Planned',
            'Roster Presented','Roster Scheduled']).cast(pl.Float64),
])
columns_to_sec = ['Time_Of_Day','Open Time','Extra Time','Break Time',
                  'Lunch Time','Training','NCNS','AL','Target']
IEX = IEX.with_columns([
    (pl.col(col).fill_null(0).cast(pl.Float64) * 3600).alias(col) for col in columns_to_sec
])
Night_Shift_1 = IEX[['Date','Email Id','Night_Shift']].unique()
Night_Shift_2 = Night_Shift_1.with_columns(
    (pl.col('Date') - pl.duration(days=1)).alias('Previous Date')
)
Night_Shift = Night_Shift_2.join(Night_Shift_1,
    left_on=['Previous Date','Email Id'], right_on=['Date','Email Id'], how='left')
Night_Shift = Night_Shift.rename({'Night_Shift_right': 'Previous_Night_Shift'})
Previous_Date = IEX[['Date','Email Id','Datetime_First_Start_Shift','Shift Tracking']].unique()

--- FULL FOLDER PATHS LIST ---
input_performance: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data
output_miv_performance: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/MIV/MIV_Data
hc_extend_by_month: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
input_survey: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN
input_afcr: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR
input_t3: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN
input_iex: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX
mapping_file: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx
global_hc: C:/Users/ADMIN/Concentrix Corporation/WFM-Ex

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_24300\2863908424.py:32: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  .str.to_datetime("%Y-%m-%d %H:%M:%S.%f", strict=False),


## Survey — New Schema (Survey Dump)

Cột mapping:
- `NPS Type` → `_nps_type`, `_promoter`, `_detractor`, `_neutral`, `_survey`
- `DUET Score Type` → `DUET` (Positive=1, Negative=0)
- `Response Text EN` → `_verbatim`

Removed (no longer in new Survey Dump):
`_ir`, `_ae`, `_offer`, `d_happy_response`, `d_surprised_response`, `e/t/u_response`, `delight`, `usability`, `ease`, `trust`

In [25]:
SURVEY_INPUT = input_data(folder_paths["input_survey"], filename_keyword="Survey Dump")

SURVEY_INPUT = SURVEY_INPUT.rename({
    "Conversation_id": "Conversation Id",
    "Agent Email":     "Agent Email ID",
})

SURVEY_INPUT = SURVEY_INPUT.with_columns(
    pl.concat_str(
        [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
        separator="_"
    ).alias("key_survey")
)

# NPS flags from NPS Type
survey_nps = (
    SURVEY_INPUT
    .filter(pl.col("NPS Type").is_not_null())
    .with_columns([
        pl.col("NPS Type").alias("_nps_type"),
        pl.when(pl.col("NPS Type").str.to_lowercase() == "promoter")
          .then(1).otherwise(0).alias("_promoter"),
        pl.when(pl.col("NPS Type").str.to_lowercase() == "detractor")
          .then(1).otherwise(0).alias("_detractor"),
        pl.when(pl.col("NPS Type").str.to_lowercase().is_in(["neutral", "passive"]))
          .then(1).otherwise(0).alias("_neutral"),
        pl.lit(1).alias("_survey"),
    ])
    .select(["key_survey", "Conversation Id", "_nps_type", "_promoter",
             "_detractor", "_neutral", "_survey"])
    .unique()
)

# DUET: Positive=1, Negative=0
survey_duet = (
    SURVEY_INPUT
    .filter(pl.col("DUET Score Type").is_not_null())
    .with_columns(
        pl.when(pl.col("DUET Score Type").str.to_lowercase().str.contains("positive"))
          .then(1).otherwise(0).alias("DUET")
    )
    .select(["key_survey", "Conversation Id", "DUET"])
    .unique()
)

# Verbatim
verbatim = (
    SURVEY_INPUT
    .filter(
        pl.col("Response Text EN").is_not_null() &
        (pl.col("Response Text EN").str.strip_chars() != "")
    )
    .with_columns(
        pl.col("Response Text EN")
          .str.replace_all(r"[\r\n\t]+", " ")
          .str.strip_chars()
          .alias("_verbatim")
    )
    .select(["key_survey", "Conversation Id", "_verbatim"])
    .unique()
)

# Merge all survey components
survey_final = (
    survey_nps
    .join(survey_duet, on=["key_survey", "Conversation Id"], how="left")
    .join(verbatim,    on=["key_survey", "Conversation Id"], how="left")
)

print(survey_final.shape)
survey_final.head(3)

(51703, 9)


key_survey,Conversation Id,_nps_type,_promoter,_detractor,_neutral,_survey,DUET,_verbatim
str,str,str,i32,i32,i32,i32,i32,str
"""johnmaged.awad@concentrix.com_…","""59bc519e-fb8e-4b0d-b390-6f6cab…","""Promoter""",1,0,0,1,1,"""Great Service from John, he wa…"
"""johnmaged.awad@concentrix.com_…","""ddda4974-95e6-458d-8c83-53188f…","""Neutral""",0,0,1,1,0,null
"""hassan.hassan5@concentrix.com_…","""10151aaf-5ad9-4ff9-b8c5-c4649e…","""Detractor""",0,1,0,1,0,"""I did not get the personalised…"


In [26]:
T3_INPUT = input_data(folder_paths["input_t3"], filename_keyword="T3_CNX_AWS")

t3_final = (
    T3_INPUT
    .with_columns(
        pl.when(
            pl.col("Transfer Destination").str.contains("Tier 3", literal=True)
        ).then(1).otherwise(0).alias("T3")
    )
    .filter(pl.col("T3") == 1)
    .with_columns(
        pl.concat_str(
            [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_t3")
    )
    .select(["key_t3", "T3"])
    .unique()
)
t3_final.head(3)

key_t3,T3
str,i32
"""ngochien.ly@concentrix.com_52c…",1
"""ankit.dhar@concentrix.com_0d82…",1
"""thiphuonghoa.hoang@concentrix.…",1


In [27]:
DELAYED_CLOSURE_INPUT = (
    input_data(folder_paths["input_delayed_closure"], filename_keyword="excess_aws")
    .unique(subset=["User Email", "Conversation ID"], keep="last")
)

delayed_closure = (
    DELAYED_CLOSURE_INPUT
    .select([
        "User Email", "Conversation ID",
        "Excess Time", "Disconnected Reason (groups)",
        "last_traveler_message_sent_datetime_utc",
    ])
    .rename({
        "User Email":      "Agent Email ID",
        "Conversation ID": "Conversation Id",
        "Excess Time":     "_excess_time_raw",
    })
    .with_columns(
        pl.col("_excess_time_raw").cast(pl.Float64).fill_null(0).alias("Exceed Time"),
    )
    .with_columns([
        (pl.col("Exceed Time") > 0).cast(pl.Int8).alias("Exceed Chat"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("agent").cast(pl.Int8).alias("Agent Disconnect"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("ghost").cast(pl.Int8).alias("Ghost"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("requeue").cast(pl.Int8).alias("Requeued"),
        pl.col("last_traveler_message_sent_datetime_utc")
          .is_null().cast(pl.Int8).alias("Traveler Unresponsive"),
        pl.concat_str(
            [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_delayed_closure"),
    ])
    .drop(["_excess_time_raw", "Disconnected Reason (groups)",
           "last_traveler_message_sent_datetime_utc"])
    .group_by(["Agent Email ID", "Conversation Id", "key_delayed_closure"])
    .agg([
        pl.sum("Exceed Time"),
        pl.sum("Exceed Chat"),
        pl.sum("Agent Disconnect"),
        pl.sum("Ghost"),
        pl.sum("Requeued"),
        pl.sum("Traveler Unresponsive"),
    ])
)

_em = pl.col("Exceed Time") / 60
delayed_closure = delayed_closure.with_columns(
    pl.when(pl.col("Exceed Time") <= 0).then(pl.lit(None, dtype=pl.Utf8))
      .when(_em <= 1).then(pl.lit("01 Mins"))
      .when(_em <= 2).then(pl.lit("02 Mins"))
      .when(_em <= 3).then(pl.lit("03 Mins"))
      .when(_em <= 4).then(pl.lit("04 Mins"))
      .when(_em <= 5).then(pl.lit("05 Mins"))
      .when(_em <= 10).then(pl.lit("05-10 Mins"))
      .when(_em <= 15).then(pl.lit("10-15 Mins"))
      .when(_em <= 30).then(pl.lit("15-30 Mins"))
      .otherwise(pl.lit("30+ Min"))
      .alias("Exceed Bucket")
)
print(delayed_closure.height)

75371


In [28]:
def process_afcr_folder(folder_path: str) -> pl.DataFrame:
    all_dataframes = []
    for file_path in glob.glob(os.path.join(folder_path, "*.csv")):
        file_name   = os.path.basename(file_path)
        export_time = datetime.fromtimestamp(os.path.getmtime(file_path)).strftime('%Y-%m-%d %H:%M:%S')
        df = pl.read_csv(
            file_path,
            encoding="utf-8",
            schema_overrides={"Itinerary": pl.String},
            infer_schema_length=10000,
            ignore_errors=True
        )
        cast_exprs = []
        for col, dtype in [("Handle Time", pl.Float64), ("Duet", pl.Float64),
                           ("Passed Sessions", pl.Int64), ("Failed Sessions", pl.Int64)]:
            if col in df.columns:
                cast_exprs.append(pl.col(col).cast(dtype, strict=False))
        if cast_exprs:
            df = df.with_columns(cast_exprs)
        df = df.with_columns([
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time')
        ])
        all_dataframes.append(df)
    if not all_dataframes:
        return pl.DataFrame()
    return pl.concat(all_dataframes, how="diagonal_relaxed")


afcr_input = process_afcr_folder(folder_paths["input_afcr"])

if not afcr_input.is_empty():
    afcr_input = (
        afcr_input
        .filter(pl.col("Vendor Partner Location") == "Concentrix (Ho Chi Minh City)")
        .select([
            pl.col("Agent Email Address").alias("Agent Email ID"),
            pl.col("Conversation Id"),
            pl.col("Passed Sessions"),
            pl.col("Failed Sessions"),
            pl.when(pl.col("Passed Sessions").is_in([0, 1])).then(1).otherwise(0).alias("Total Sessions")
        ])
        .unique()
    )

In [29]:
RE_DIRECT_INPUT = input_data(folder_paths["input_csv_re_direct"])

re_direct_final = (
    RE_DIRECT_INPUT
    .with_columns(
        Re_Direct=pl.lit(1),
        **{
            "Re-Direct Text": (
                pl.col("Text").cast(pl.Utf8).fill_null("")
                  .str.replace_all(r"(\r\n|\r|\n)+", " | ")
                  .str.replace_all(r"[\-•\u2022\u25CF\u25E6\u2043\u2219\u00B7\u2013\u2014]+", "")
                  .str.replace_all(r"\s{2,}", " ")
                  .str.strip_chars()
            )
        }
    )
    .with_columns(
        pl.concat_str(
            [pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_redirect")
    )
    .select(["key_redirect", "Agent People Id", "Re_Direct", "Re-Direct Text"])
    .unique(subset=["key_redirect"], maintain_order=True)
)

In [30]:
PERFORMANCE_INPUT = input_data(
    folder_paths["input_performance"],
    filename_keyword="aws_performance_retail_rawdata"
).filter(pl.col( "Agent Vendor Location",).str.contains("Ho Chi Minh", literal=True))

try:
    if PERFORMANCE_INPUT.columns[0] == "":
        PERFORMANCE_INPUT = PERFORMANCE_INPUT.drop(PERFORMANCE_INPUT.columns[0])
except: pass

print(PERFORMANCE_INPUT.columns)

# ── Joined Time parser ─────────────────────────────────────────────────────
def _build_joined_time(time_col: str = "Connected To Agent Time") -> pl.Expr:
    raw = pl.col(time_col).cast(pl.Utf8).str.strip_chars()
    return (
        pl.coalesce([
            raw.str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
            raw.str.strptime(pl.Datetime, "%m/%d/%Y %H:%M",    strict=False),
        ])
        .alias("Joined Time")
    )

# ── Duration parser (seconds or HH:MM:SS) ─────────────────────────────────
def _duration_to_seconds(col: str) -> pl.Expr:
    raw       = pl.col(col).cast(pl.Utf8).str.strip_chars()
    as_number = raw.str.replace_all(",", "").cast(pl.Float64, strict=False)
    h = raw.str.extract(r"^(\d+):\d{2}:\d{2}$", 1).cast(pl.Float64, strict=False)
    m = raw.str.extract(r"^\d+:(\d{2}):\d{2}$", 1).cast(pl.Float64, strict=False)
    s = raw.str.extract(r"^\d+:\d{2}:(\d{2})$", 1).cast(pl.Float64, strict=False)
    from_hhmmss = h * 3600 + m * 60 + s
    return pl.when(as_number.is_not_null()).then(as_number).otherwise(from_hhmmss).alias(col)

# ── AWS column adapter ─────────────────────────────────────────────────────
PERFORMANCE_INPUT = (
    PERFORMANCE_INPUT
    .with_columns(_build_joined_time())
    .rename({
        "Handle Time":                    "Handle Time (Sum)",
        "Talk Time":                      "Talk Time (Sum)",
        "Acw Duration":                   "Wrap Up Time (Sum)",
        "Agent Vendor Location":          "Agent Business Location",
        "Outbound Initiated (Yes / No)":  "Initiated Outbound (Yes / No)",
        "Product":                        "Latest VA Product",
        "Intent":                         "Latest VA Intent",
        "Locale":                         "Language",
    })
    .with_columns([
        pl.col("Latest VA Product").fill_null("UNKNOWN").alias("Latest VA Product"),
        pl.col("Latest VA Intent").fill_null("UNKNOWN").alias("Latest VA Intent"),
    ])
)

print(PERFORMANCE_INPUT.select(["Connected To Agent Time", "Joined Time"]).head(5))

existing_cols = set(PERFORMANCE_INPUT.columns)

columns_to_cast = {
    "Handle Time (Sum)":  pl.Float64,
    "Talk Time (Sum)":    pl.Float64,
    "Wrap Up Time (Sum)": pl.Float64,
    "Hold Time (Sum)":    pl.Float64,
    "Handle (Count)":     pl.Int64,
}

casts = []
for _col, _dtype in columns_to_cast.items():
    if _col not in existing_cols:
        continue
    if _dtype == pl.Float64:
        casts.append(_duration_to_seconds(_col))
    else:
        casts.append(pl.col(_col).cast(_dtype, strict=False).alias(_col))

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_INPUT.with_columns(casts)

# ── Routing Profile → LOB + Agent Queue Group Name override ───────────────
LG_CHAT_PROFILES = [
    "Chat_AC_GLB_EN_Car_Activity",
    "Chat_AC_GLB_EN_Lodging_Nesting",
    "Chat_AC_GLB_EN_Lodging_Proficient",
]
NL_CHAT_PROFILES = [
    "Chat_AC_GLB_EN_Proficient",
    "Chat_AC_GLB_EN_NL_Nesting",
]

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.col("Agent Routing Profile Name").alias("Agent Queue Group Name"),
    pl.when(pl.col("Agent Routing Profile Name").is_in(LG_CHAT_PROFILES))
      .then(pl.lit("LG Chat"))
      .when(pl.col("Agent Routing Profile Name").is_in(NL_CHAT_PROFILES))
      .then(pl.lit("NL Chat"))
      .otherwise(pl.col("Agent Routing Profile Name"))
      .alias("LOB"),
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.col("Joined Time").dt.date().alias("Joined Date")
])
PERFORMANCE_NEXT_STEP = PERFORMANCE_NEXT_STEP.with_columns([
    (pl.col("Joined Time") + pl.duration(hours=14)).alias("Join Time (VNT)"),
    (pl.col("Joined Time") + pl.duration(hours=14)).dt.date().alias("Join Date (VNT)")
])

['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Vendor Location', 'Outbound Initiated (Yes / No)', 'Business Segment Name', 'Partner Name', 'Locale', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time', 'Talk Time', 'Assigned Agent Time', 'Acw Duration', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disconnect Reason', 'Requeue Time', 'Inbound Message Count', 'Outbound Message Count', 'Transfer Initiated (Yes / No)', 'Agent Tenure in Days', 'Initial Channel Type', 'Answer Time', 'Queue Time', 'Intent', 'Product', 'Contact Disconnect (Count)', 'Handle (Count)', 'Hold Time (Sum)', 'File Name', 'Export Time']
shape: (5, 2)
┌─────────────────────────┬─────────────────────┐
│ Connected To Agent Time ┆ Joined Time         │
│ ---                     ┆ ---                 │
│ str                     ┆ datetime[μs]        │
╞════════════════

In [31]:
HC_MASTER_DATABASE = input_data(folder_paths["hc_extend_by_month"])
HC_MASTER_DATABASE = HC_MASTER_DATABASE.rename({'Date Start Week': 'Week_Monday'})
HC_MASTER_DATABASE = HC_MASTER_DATABASE.with_columns([
    pl.col('Date').str.strptime(pl.Date, "%Y-%m-%d", strict=False)
])

hc_master_selected = HC_MASTER_DATABASE.select([
    "Date","Email Id","OracleID","People ID","IEX ID","Employee Name","Alias","Designation",
    "Detail Status","Active","TL ID","Supervisor Email",
    "Supervisor Name","Wave","LOB",'LG Tenure','NL Tenure',
    'Mini TL - Email','Mini TL - Short Name','Mini TL Start Date','Site'
]).unique()

hc_master_selected = hc_master_selected.rename({'Mini TL - Short Name': 'Mini TL', 'LOB': 'Group'})

performance_merged = PERFORMANCE_NEXT_STEP.join(
    hc_master_selected,
    left_on=["Joined Date","Agent Email ID"],
    right_on=["Date","Email Id"],
    how="left"
)

GLOBAL_HC = pl.read_parquet(folder_paths["global_hc"])
global_hc_clean = GLOBAL_HC.select(["SSO ID","Production Start date","Agent/Non Agent"]).unique(subset=["SSO ID"], keep="first")
merged_global_hc = performance_merged.join(global_hc_clean, left_on="Agent Email ID", right_on="SSO ID", how="left")

merged_iex = merged_global_hc.join(
    IEX[['Date','Email Id','First Shift','Datetime_First_Start_Shift','Night_Shift']],
    left_on=['Join Date (VNT)','Agent Email ID'],
    right_on=['Date','Email Id'],
    how='left'
)

mapping   = pl.read_excel(folder_paths["mapping_file"])
lc_mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")
performance_cleaned = merged_iex.join(mapping, on="Agent Queue Group Name", how='left')

Could not determine dtype for column 5, falling back to string


In [32]:
performance_updated_ns = performance_cleaned.with_columns(
    (pl.col('Join Date (VNT)') - pl.duration(days=1)).alias('Previous Date')
)
performance_updated_ns = performance_updated_ns.join(
    Night_Shift[['Date','Email Id','Night_Shift','Previous_Night_Shift']],
    left_on=['Join Date (VNT)','Agent Email ID'],
    right_on=['Date','Email Id'],
    how='left'
)

def update_night_shift(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.when(
            (pl.col('Night_Shift') == 0) &
            (pl.col('Join Time (VNT)').dt.time() < pl.time(17, 0)) &
            (pl.col('Join Time (VNT)').dt.time() >= pl.time(0, 0)) &
            (pl.col('Previous_Night_Shift') == 1)
        ).then(False).otherwise(True).alias('Night_Shift_2_Check')
    )
    df = df.with_columns(
        pl.when(pl.col('Night_Shift_2_Check') == False)
          .then(1).otherwise(pl.col('Night_Shift')).alias('Night_Shift')
    )
    df = df.with_columns(
        pl.when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 12) & (pl.col('Previous_Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        .when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 12) & (pl.col('Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        .when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 18) & (pl.col('Night_Shift') == 0))
          .then(pl.col('Join Date (VNT)'))
        .when((pl.col('Join Time (VNT)').dt.hour() >= 18) & (pl.col('Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)'))
        .otherwise(pl.col('Join Date (VNT)'))
        .alias('_Date_Converted')
    )
    return df

performance_updated_ns = update_night_shift(performance_updated_ns)
performance_updated_ns.select(["Joined Time","Join Time (VNT)","Join Date (VNT)","_Date_Converted"]).head(5)

Joined Time,Join Time (VNT),Join Date (VNT),_Date_Converted
datetime[μs],datetime[μs],date,date
2026-06-30 09:07:53,2026-06-30 23:07:53,2026-06-30,2026-06-30
2026-06-30 11:12:59,2026-07-01 01:12:59,2026-07-01,2026-06-30
2026-06-30 08:50:13,2026-06-30 22:50:13,2026-06-30,2026-06-30
2026-06-30 00:11:16,2026-06-30 14:11:16,2026-06-30,2026-06-30
2026-06-30 12:57:22,2026-07-01 02:57:22,2026-07-01,2026-06-30


In [33]:
# NOTE: _promoter/_detractor/_neutral/_survey/_nps_type now come from survey_final join.
# CCR72/_fup_72/_rr/_offer/_ir/_ae removed — no source columns in new schema.
performance_processed = performance_updated_ns.with_columns([
    pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0).alias("_aob"),

    pl.col("Joined Time").dt.date().alias("_PST.Date"),
    pl.col("Joined Time").dt.strftime("%y_%m").alias("_PST.Month"),
    pl.col("Joined Time").dt.week().alias("_PST.Week"),
    pl.col("Joined Time").dt.year().alias("_PST.Year"),

    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null(""),
        pl.col("Joined Time").dt.strftime("%y%m%d%H%M%S")
    ], separator="_").alias("_conver_unique"),

    pl.when(pl.col("Group").is_in(["Non_Lodging", "Lodging"]))
      .then(pl.lit("agent")).otherwise(None).alias("Agent"),
    pl.col("_Date_Converted").alias("_Date"),
    pl.when(pl.col("Group") == "Lodging").then(pl.lit("LG Tenure"))
      .when(pl.col("Group") == "Non_Lodging").then(pl.lit("NL Tenure"))
      .otherwise(None).alias("Tenure"),
])

# AON Days + Status
performance_processed = performance_processed.with_columns(
    (
        pl.col("_PST.Date").cast(pl.Date) -
        pl.col("Production Start date").cast(pl.String)
          .str.to_date("%Y-%m-%d %H:%M:%S", strict=False).cast(pl.Date)
    ).dt.total_days().cast(pl.Int32).alias("AON_Days")
)
performance_processed = performance_processed.with_columns([
    pl.when(
        pl.col("AON_Days").is_null() &
        (pl.col("Agent/Non Agent").is_in(["Agent","ID Deleted"]))
    ).then(pl.lit("Nesting"))
      .when(pl.col("AON_Days") > 180).then(pl.lit("> 180 Days"))
      .when(pl.col("AON_Days") >= 91).then(pl.lit("91 - 180"))
      .when(pl.col("AON_Days") >= 61).then(pl.lit("61 - 90"))
      .when(pl.col("AON_Days") >= 31).then(pl.lit("31 - 60"))
      .when(pl.col("AON_Days") >= 0).then(pl.lit("00 - 30"))
      .otherwise(None).alias("AON Status")
])

# LC threshold
performance_processed = performance_processed.join_asof(
    lc_mapping, left_on="_PST.Date", right_on="Effective Date", by="LOB", strategy="backward"
)
performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") >= pl.col("Threshole_LC")).cast(pl.Int8).alias("_lc"),
    (pl.col("Handle Time (Sum)") < 240).cast(pl.Int8).alias("Short Chat"),
])

# Composite keys
performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID"), pl.col("_PST.Date").dt.strftime("%y%m%d")]).alias("KEY"),
    pl.concat_str([pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("EmailID_ConversationID_KEY"),
    pl.concat_str([pl.col("OracleID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("OracleID_ConversationID_KEY"),
    pl.concat_str([pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("PeopleID_ConversationID_KEY"),
])

# 30-min interval + period (native Polars, no map_elements)
_pst_hour = pl.col("Joined Time").dt.hour()
_pst_min  = pl.col("Joined Time").dt.minute()
_vnt_hour = pl.col("Join Time (VNT)").dt.hour()
_vnt_min  = pl.col("Join Time (VNT)").dt.minute()
_shift_h  = pl.col("Datetime_First_Start_Shift").dt.hour()

def _interval(h, m):
    sm = pl.when(m < 30).then(pl.lit(0)).otherwise(pl.lit(30))
    em = pl.when(m < 30).then(pl.lit(29)).otherwise(pl.lit(59))
    return pl.concat_str([
        h.cast(pl.Utf8).str.zfill(2), pl.lit(":"),
        sm.cast(pl.Utf8).str.zfill(2), pl.lit("-"),
        h.cast(pl.Utf8).str.zfill(2), pl.lit(":"),
        em.cast(pl.Utf8).str.zfill(2),
    ])

performance_processed = performance_processed.with_columns([
    _interval(_pst_hour, _pst_min).alias("_PST.Interval"),
    _interval(_vnt_hour, _vnt_min).alias("_VNT.Interval"),
    pl.when(_shift_h >= 18).then(pl.lit("Night"))
      .when(_shift_h >= 12).then(pl.lit("Mid"))
      .otherwise(pl.lit("Morning")).alias("_VNT.Period"),
])

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_24300\4166081039.py:47: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  performance_processed = performance_processed.join_asof(


In [34]:
performance_merged_survey_t3 = (
    performance_processed
    .join(t3_final,                                    left_on="_conver_unique",              right_on="key_t3",               how="left")
    .join(delayed_closure,                             left_on="EmailID_ConversationID_KEY",  right_on="key_delayed_closure",  how="left")
    .join(re_direct_final,                             left_on="PeopleID_ConversationID_KEY", right_on="key_redirect",         how="left")
    .join(survey_final.drop("Conversation Id"),        left_on="EmailID_ConversationID_KEY",  right_on="key_survey",           how="left")
    .join(afcr_input,                                  on=["Agent Email ID", "Conversation Id"],                               how="left")
)

print(performance_merged_survey_t3.columns)
print(performance_merged_survey_t3.select(["_PST.Date","Production Start date","AON_Days"]).head())

selected_columns = [
    "Export Time","File Name","Agent People Id","Business Segment Name","Partner Name",
    "Response Count","Response Time","Latest VA Product","Language","Latest VA Intent","Conversation Id",
    "Agent Queue Group Name","Joined Time","_PST.Interval","Agent Email ID",
    "Handle (Count)","Handle Time (Sum)","Hold Time (Sum)","Talk Time (Sum)",
    "Join Time (VNT)","_VNT.Interval","_VNT.Period","Wrap Up Time (Sum)","Agent Business Location",
    "_PST.Date","_PST.Month","_PST.Year","_aob","LOB","_conver_unique",
    "_nps_type","_promoter","_detractor","_neutral","_survey",
    "DUET","_verbatim",
    "T3","Re_Direct","Re-Direct Text",
    "Exceed Time","Exceed Chat","Exceed Bucket",
    "Agent Disconnect","Ghost","Requeued","Traveler Unresponsive",
    "_PST.Week","_lc","AON Status","Agent/Non Agent","Tenure",
    "OracleID","People ID","IEX ID","Employee Name","Alias","Designation",
    "Detail Status","Active","TL ID","Supervisor Email","Supervisor Name",
    "Wave","Group","_Date","Mini TL - Email","Mini TL","Mini TL Start Date","Site",
    "Short Chat","Passed Sessions","Failed Sessions","Total Sessions",
]

fcr_columns = [
    "LOB","OracleID_ConversationID_KEY","EmailID_ConversationID_KEY",
    "_nps_type","_detractor","_survey","_PST.Week","AON Status",
    "Agent/Non Agent","Tenure","Employee Name","Alias","Designation","Detail Status",
    "TL ID","Supervisor Name","Supervisor Email","Wave","Group","_Date",
    "Mini TL - Email","Mini TL","Mini TL Start Date","Site","Agent Business Location",
]

missing_cols = [col for col in selected_columns if col not in performance_merged_survey_t3.columns]
print("Missing columns:", missing_cols)

performance_filtered = performance_merged_survey_t3.select(selected_columns).unique()

performance_filtered = performance_filtered.sort(
    by=["Conversation Id","Agent Email ID","Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id","Agent Email ID"]) > 1)
    .cast(pl.Int8).alias("Duplicate_Flag")
)

['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Business Segment Name', 'Partner Name', 'Language', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time (Sum)', 'Talk Time (Sum)', 'Assigned Agent Time', 'Wrap Up Time (Sum)', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disconnect Reason', 'Requeue Time', 'Inbound Message Count', 'Outbound Message Count', 'Transfer Initiated (Yes / No)', 'Agent Tenure in Days', 'Initial Channel Type', 'Answer Time', 'Queue Time', 'Latest VA Intent', 'Latest VA Product', 'Contact Disconnect (Count)', 'Handle (Count)', 'Hold Time (Sum)', 'File Name', 'Export Time', 'Joined Time', 'LOB', 'Joined Date', 'Join Time (VNT)', 'Join Date (VNT)', 'OracleID', 'People ID', 'IEX ID', 'Employee Name', 'Alias', 'Designation', 'Detail Status', 'Active', 'TL ID', 'S

In [35]:
performance_filtered = performance_filtered.sort(
    by=["Conversation Id","Agent Email ID","Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id","Agent Email ID"]) > 1)
    .cast(pl.Int8).alias("IDs Removed")
)

In [36]:
# -------------------------------------------------------------------------------------
# DUET dedup: if a Conversation Id has >3 rows with DUET=0,
# keep only the 1st as 0 and nullify the rest.
# Logic unchanged — DUET is now binary (0/1) from DUET Score Type.
# -------------------------------------------------------------------------------------
performance_filtered = performance_filtered.sort(["Conversation Id"])

performance_filtered = performance_filtered.with_columns([
    (pl.col("DUET") == 0).alias("_is_zero")
])

performance_filtered = performance_filtered.with_columns([
    pl.col("_is_zero").sum().over("Conversation Id").alias("_zero_total"),
    pl.col("_is_zero").cum_sum().over("Conversation Id").alias("_zero_rank"),
])

performance_filtered = performance_filtered.with_columns(
    pl.when(
        (pl.col("_is_zero")) &
        (pl.col("_zero_total") > 3) &
        (pl.col("_zero_rank") > 1)
    ).then(None)
     .otherwise(pl.col("DUET"))
     .alias("DUET")
).drop(["_is_zero","_zero_total","_zero_rank"])

In [42]:
import pandas as pd
import numpy as np
import warnings

_avail = set(performance_filtered.columns)

DIM_MIV_PATH = f"{os.path.expanduser('~').replace(chr(92), '/')}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/MIV/dim_miv.xlsx"
ATD_PATH     = f"{os.path.expanduser('~').replace(chr(92), '/')}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/ATD_Final.parquet"

def _parse_dim_section(raw, col_start, col_end, col3, col4):
    out = raw.iloc[1:, col_start:col_end+1].copy()
    out.columns = ["Eff_Date", "LOB", "Metric", col3, col4]
    out = out.dropna(subset=["LOB", "Metric"]).copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out["Eff_Date"] = pd.to_datetime(out["Eff_Date"], errors="coerce")
    out[col3] = pd.to_numeric(out[col3], errors="coerce")
    out[col4] = pd.to_numeric(out[col4], errors="coerce")
    return out.dropna(subset=["Eff_Date"]).reset_index(drop=True)

_raw_dim     = pd.read_excel(DIM_MIV_PATH, sheet_name="dim", header=None)
AGENT_SCORE  = _parse_dim_section(_raw_dim,  3,  7, "Range",  "Score")
AGENT_WEIGHT = _parse_dim_section(_raw_dim,  9, 13, "Target", "Weight")
AS_OF        = pd.Timestamp("2026-07-31")

_sme_raw = pd.read_excel(DIM_MIV_PATH, sheet_name="sme")
_sme_raw.columns = _sme_raw.columns.str.strip()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _sme_raw["Eff_Date"] = pd.to_datetime(_sme_raw["Eff_Date"], errors="coerce")
_sme_raw = _sme_raw.dropna(subset=["Supervisor Email", "SME"])
sme_dim  = (_sme_raw[_sme_raw["Eff_Date"] <= AS_OF].sort_values("Eff_Date")
             .groupby("Supervisor Email", as_index=False).last()[["Supervisor Email", "SME"]])

_qa_raw         = pd.read_excel(DIM_MIV_PATH, sheet_name="qa_score")
_qa_raw.columns = _qa_raw.columns.str.strip()
_qa_month_col   = next((c for c in _qa_raw.columns if "month" in c.lower()), None)
_qa_email_col   = next((c for c in _qa_raw.columns if "email" in c.lower()), None)
_qa_score_col   = next((c for c in _qa_raw.columns if "qa" in c.lower() and "score" in c.lower()), None)
qa_dim          = _qa_raw[[_qa_month_col, _qa_email_col, _qa_score_col]].copy()
qa_dim.columns  = ["_PST.Month", "Agent Email ID", "QA_Score_month"]
qa_dim["_PST.Month"]     = qa_dim["_PST.Month"].astype(str).str.strip()
qa_dim["QA_Score_month"] = pd.to_numeric(qa_dim["QA_Score_month"], errors="coerce")

atd_raw           = pd.read_parquet(ATD_PATH)
atd_raw["Month"]  = pd.to_datetime(atd_raw["Month"], format="%b-%y", errors="coerce").dt.strftime("%y_%m")
_atd_email_col    = next((c for c in atd_raw.columns if c == "Email Id"), next((c for c in atd_raw.columns if "email" in c.lower()), None))
atd_raw["Unplanned"]   = pd.to_numeric(atd_raw["Unplanned"],   errors="coerce")
atd_raw["HC Schedule"] = pd.to_numeric(atd_raw["HC Schedule"], errors="coerce")
_atd_agg = {"Unplanned": "sum", "HC Schedule": "sum"}
if "LOB" in atd_raw.columns: _atd_agg["LOB"] = "first"
atd_dim = (atd_raw.groupby(["Month", _atd_email_col], as_index=False)
           .agg(_atd_agg)
           .rename(columns={"Month": "_PST.Month", _atd_email_col: "Agent Email ID",
                             "Unplanned": "HC_Unplanned_month", "HC Schedule": "HC_Schedule_month"}))
atd_dim["Unplanned_pct_month"] = np.where(atd_dim["HC_Schedule_month"] > 0,
    atd_dim["HC_Unplanned_month"] / atd_dim["HC_Schedule_month"], np.nan)
atd_dim["_PST.Month"] = atd_dim["_PST.Month"].astype(str).str.strip()

_hc_pd          = HC_MASTER_DATABASE.to_pandas()
_hc_pd.columns  = _hc_pd.columns.str.strip()
_att_col        = next((c for c in _hc_pd.columns if c == "ATT/Movement"),
                  next((c for c in _hc_pd.columns if "movement" in c.lower() and "lwd" not in c.lower()), None))
_lwd_col        = next((c for c in _hc_pd.columns if c == "LWD/Movement"), None)
_email_col_hc   = next((c for c in _hc_pd.columns if c == "Email Id"), next((c for c in _hc_pd.columns if "email" in c.lower()), None))
_hc_pd[_att_col] = pd.to_numeric(_hc_pd[_att_col], errors="coerce")
_hc_pd           = _hc_pd[_hc_pd[_email_col_hc].astype(str).str.contains("@", na=False)].copy()
_hc_pd["Month"]  = pd.to_datetime(_hc_pd["Month"], format="%b-%y", errors="coerce").dt.strftime("%y_%m")
if _lwd_col: _hc_pd[_lwd_col] = pd.to_datetime(_hc_pd[_lwd_col], errors="coerce")
_hc_agg_dict     = {_att_col: "sum", **({_lwd_col: "max"} if _lwd_col else {})}
hc_att_dim       = (_hc_pd.groupby(["Month", _email_col_hc], as_index=False).agg(_hc_agg_dict)
                    .rename(columns={"Month": "_PST.Month", _email_col_hc: "Agent Email ID",
                                     _att_col: "ATT_Movement_month",
                                     **({_lwd_col: "LWD_Movement_month"} if _lwd_col else {})}))
hc_att_dim["_PST.Month"] = hc_att_dim["_PST.Month"].astype(str).str.strip()

def _build_score_lk(df):
    lk = {}
    for _, r in df.iterrows():
        lk.setdefault((r["LOB"], r["Metric"]), {}).setdefault(r["Eff_Date"], []).append((float(r["Range"]), float(r["Score"])))
    return lk

def _build_weight_lk(df):
    lk = {}
    for _, r in df.iterrows():
        lk.setdefault((r["LOB"], r["Metric"]), {})[r["Eff_Date"]] = (
            float(r["Target"]) if pd.notna(r["Target"]) else None,
            float(r["Weight"]) if pd.notna(r["Weight"]) else 0.0)
    return lk

def _latest(mapping, as_of):
    valid = {d: v for d, v in mapping.items() if d <= as_of}
    return valid[max(valid)] if valid else None

def lk_tw(wt_lk, lob, metric, as_of):
    val = _latest(wt_lk.get((lob, metric), {}), as_of)
    return val if val else (None, 0.0)

def lk_score(sc_lk, lob, metric, actual, as_of):
    if actual is None or (isinstance(actual, float) and np.isnan(actual)): return None
    rs = _latest(sc_lk.get((lob, metric), {}), as_of)
    if not rs: return None
    eligible = [s for r, s in rs if r <= actual]
    if not eligible: return None
    sorted_rs = sorted(rs, key=lambda x: x[0])
    hib = sorted_rs[-1][1] >= sorted_rs[0][1]
    return int(max(eligible) if hib else min(eligible))

def lk_att(actual, target, hib=True):
    if actual is None or target is None or target == 0: return None
    if isinstance(actual, float) and np.isnan(actual): return None
    if not hib and actual == 0: return 1.0
    return min(1.0, actual / target if hib else target / actual)

sc_lk = _build_score_lk(AGENT_SCORE)
wt_lk = _build_weight_lk(AGENT_WEIGHT)

def _find_col(candidates, avail):
    for c in candidates:
        if c in avail: return c
    return None

COL_PEOPLE = _find_col(["People ID", "Agent People Id", "PeopleID"], _avail)
COL_EXCEED = _find_col(["Exceed Chat"], _avail)
COL_FCR_P  = _find_col(["Passed Sessions"], _avail)
COL_FCR_F  = _find_col(["Failed Sessions"], _avail)
COL_FCR_T  = _find_col(["Total Sessions"], _avail)
COL_DUP    = _find_col(["Duplicate_Flag", "IDs Removed"], _avail)
GROUP      = [COL_PEOPLE, "_PST.Month", "LOB"]

_aht_agg = (performance_filtered
    .group_by(["_conver_unique"] + GROUP, maintain_order=True)
    .agg(pl.col("Handle Time (Sum)").max().alias("_ch"))
    .group_by(GROUP, maintain_order=True)
    .agg(pl.col("_ch").mean().alias("aht")))

_exc_agg = (performance_filtered.unique(subset=[COL_PEOPLE, "_conver_unique"])
    .group_by(GROUP, maintain_order=True)
    .agg(pl.col(COL_EXCEED).cast(pl.Float64, strict=False).sum().alias("_sum_exc")
         if COL_EXCEED else pl.lit(None).cast(pl.Float64).alias("_sum_exc")))

_agg_exprs = [
    pl.col("_conver_unique").n_unique().alias("chat_count"),
    pl.col("_lc").mean().alias("lc"), pl.col("_lc").sum().alias("_lc_sum"),
    pl.col("_promoter").sum().alias("_prom"), pl.col("_detractor").sum().alias("_detr"), pl.col("_survey").sum().alias("_surv"),
]
for _c in ["Employee Name", "Supervisor Name", "Supervisor Email", "Wave", "OracleID", "Agent Email ID", "Designation", "Detail Status", "Group", "Active", "TL ID"]:
    if _c in _avail: _agg_exprs.append(pl.col(_c).first())
if "_PST.Date" in _avail: _agg_exprs.append(pl.col("_PST.Date").min())
if "_Date"     in _avail: _agg_exprs.append(pl.col("_Date").min().alias("_VN.Date"))
if "_neutral"  in _avail: _agg_exprs.append(pl.col("_neutral").sum().alias("_Neutral"))
if COL_DUP:               _agg_exprs.append(pl.col(COL_DUP).cast(pl.Float64, strict=False).sum().alias("Duplicate_Flag"))
if COL_EXCEED:            _agg_exprs.append(pl.col(COL_EXCEED).cast(pl.Float64, strict=False).sum().alias("_exceed_sum"))
_agg_exprs.append(pl.lit(None).cast(pl.Float64).alias("qa"))
_agg_exprs.append(pl.col(COL_FCR_P).cast(pl.Float64, strict=False).sum().alias("_fcr_p") if COL_FCR_P else pl.lit(0.0).alias("_fcr_p"))
_agg_exprs.append(pl.col(COL_FCR_F).cast(pl.Float64, strict=False).sum().alias("_fcr_f") if COL_FCR_F else pl.lit(0.0).alias("_fcr_f"))
if COL_FCR_T:   _agg_exprs.append(pl.col(COL_FCR_T).cast(pl.Float64, strict=False).sum().alias("_fcr_t"))
elif COL_FCR_F: _agg_exprs.append((pl.col(COL_FCR_P).cast(pl.Float64, strict=False).sum() + pl.col(COL_FCR_F).cast(pl.Float64, strict=False).sum()).alias("_fcr_t"))
else:           _agg_exprs.append(pl.lit(0.0).alias("_fcr_t"))

miv_raw = (performance_filtered.group_by(GROUP, maintain_order=True).agg(_agg_exprs)
           .join(_aht_agg, on=GROUP, how="left").join(_exc_agg, on=GROUP, how="left")).to_pandas()

if COL_PEOPLE != "People ID": miv_raw = miv_raw.rename(columns={COL_PEOPLE: "People ID"})
miv_raw["_PST.Month"] = miv_raw["_PST.Month"].astype(str).str.strip()
miv_raw = miv_raw.merge(sme_dim, on="Supervisor Email", how="left") if "Supervisor Email" in miv_raw.columns else miv_raw.assign(SME=None)

if "Agent Email ID" in miv_raw.columns:
    miv_raw = miv_raw.merge(qa_dim, on=["_PST.Month", "Agent Email ID"], how="left")
    miv_raw = miv_raw.merge(atd_dim[["_PST.Month", "Agent Email ID", "HC_Unplanned_month", "HC_Schedule_month", "Unplanned_pct_month"]], on=["_PST.Month", "Agent Email ID"], how="left")
    miv_raw = miv_raw.merge(hc_att_dim, on=["_PST.Month", "Agent Email ID"], how="left")
    miv_raw["qa"] = miv_raw["QA_Score_month"] / 100
else:
    for _col in ["QA_Score_month", "HC_Unplanned_month", "HC_Schedule_month", "Unplanned_pct_month", "ATT_Movement_month", "LWD_Movement_month", "qa"]:
        miv_raw[_col] = None

# ── Filler rows: agents in ATD/HC but not in performance ────────────────────
_scope_months   = set(miv_raw["_PST.Month"].astype(str).unique())
_existing_keys  = set(zip(miv_raw["_PST.Month"].astype(str), miv_raw["Agent Email ID"].astype(str)))

_hc_ref = (_hc_pd[_hc_pd["Month"].isin(_scope_months)].copy()
           .rename(columns={"Month": "_PST.Month", _email_col_hc: "Agent Email ID"}))

_atd_ref = (atd_raw[atd_raw["Month"].isin(_scope_months)][[_atd_email_col, "Month"]]
            .rename(columns={"Month": "_PST.Month", _atd_email_col: "Agent Email ID"})
            .drop_duplicates())

_all_ref = (pd.concat([_hc_ref[["_PST.Month", "Agent Email ID"]], _atd_ref], ignore_index=True)
            .drop_duplicates())
_all_ref = _all_ref[_all_ref["Agent Email ID"].astype(str).str.contains("@", na=False)]

_missing = _all_ref[~_all_ref.apply(
    lambda r: (str(r["_PST.Month"]), str(r["Agent Email ID"])) in _existing_keys, axis=1)].copy()

if len(_missing) > 0:
    _lob_map   = {"Non_Lodging": "NL Chat", "Lodging": "LG Chat"}
    _hc_meta_cols = [c for c in ["_PST.Month", "Agent Email ID", "People ID", "OracleID",
                                   "Employee Name", "Designation", "Detail Status", "Group",
                                   "Active", "TL ID", "Supervisor Email", "Supervisor Name", "Wave"]
                     if c in _hc_ref.columns]
    _hc_meta   = (_hc_ref[_hc_meta_cols].drop_duplicates(subset=["_PST.Month", "Agent Email ID"], keep="last"))

    _missing   = _missing.merge(_hc_meta, on=["_PST.Month", "Agent Email ID"], how="left")
    if "Group" in _missing.columns:
        _missing["LOB"] = _missing["Group"].map(_lob_map)

    _missing   = _missing.merge(atd_dim[["_PST.Month", "Agent Email ID", "HC_Unplanned_month", "HC_Schedule_month", "Unplanned_pct_month"]], on=["_PST.Month", "Agent Email ID"], how="left")
    _missing   = _missing.merge(hc_att_dim, on=["_PST.Month", "Agent Email ID"], how="left")
    _missing   = _missing.merge(sme_dim, on="Supervisor Email", how="left") if "Supervisor Email" in _missing.columns else _missing
    _missing   = _missing.merge(qa_dim, on=["_PST.Month", "Agent Email ID"], how="left")
    _missing["qa"] = _missing["QA_Score_month"] / 100 if "QA_Score_month" in _missing.columns else None

    for _pc in ["chat_count", "lc", "_lc_sum", "_prom", "_detr", "_surv", "aht", "_sum_exc", "_fcr_p", "_fcr_f", "_fcr_t"]:
        _missing[_pc] = None

    _missing["key_unique_month"] = _missing["_PST.Month"].astype(str) + "_" + _missing["Agent Email ID"].astype(str)
    miv_raw = pd.concat([miv_raw, _missing], ignore_index=True, sort=False)
    miv_raw["_PST.Month"] = miv_raw["_PST.Month"].astype(str).str.strip()
    print(f"Added {len(_missing)} filler rows (no volume agents)")

# ── Derived metrics ──────────────────────────────────────────────────────────
_surv_n  = pd.to_numeric(miv_raw["_surv"],       errors="coerce").fillna(0)
_prom_n  = pd.to_numeric(miv_raw["_prom"],       errors="coerce").fillna(0)
_detr_n  = pd.to_numeric(miv_raw["_detr"],       errors="coerce").fillna(0)
_chats_n = pd.to_numeric(miv_raw["chat_count"],  errors="coerce").fillna(0)
_exc_n   = pd.to_numeric(miv_raw["_sum_exc"],    errors="coerce").fillna(0)
_fcrp_n  = pd.to_numeric(miv_raw["_fcr_p"],      errors="coerce").fillna(0)
_fcrt_n  = pd.to_numeric(miv_raw["_fcr_t"],      errors="coerce").fillna(0)

miv_raw["nps"] = np.where(_surv_n  > 0, np.divide(_prom_n - _detr_n, _surv_n,  where=_surv_n  > 0, out=np.full(len(miv_raw), np.nan)), np.nan)
miv_raw["exc"] = np.where(_chats_n > 0, np.divide(_exc_n,             _chats_n, where=_chats_n > 0, out=np.full(len(miv_raw), np.nan)), np.nan)
miv_raw["fcr"] = np.where(_fcrt_n  > 0, np.divide(_fcrp_n,            _fcrt_n,  where=_fcrt_n  > 0, out=np.full(len(miv_raw), np.nan)) * 100, np.nan)

METRICS = [("fcr", "FCR", True), ("nps", "NPS", True), ("qa", "AWF", True), ("lc", "LC", False), ("aht", "AHT", False), ("exc", "EXC", False)]
lob_score = np.zeros(len(miv_raw))
point     = np.zeros(len(miv_raw))

for col, metric, hib in METRICS:
    tars, wts, atts, scs, wss = [], [], [], [], []
    for _, row in miv_raw.iterrows():
        lob = row.get("LOB")
        tar, w = lk_tw(wt_lk, lob, metric, AS_OF) if lob else (None, 0.0)
        att = lk_att(row.get(col), tar, hib)
        sc  = lk_score(sc_lk, lob, metric, row.get(col), AS_OF) if lob else None
        ws  = (sc * w) if sc is not None else 0.0
        tars.append(tar); wts.append(w); atts.append(att); scs.append(sc); wss.append(ws)
    miv_raw[f"{metric}_tar"] = tars
    miv_raw[f"{metric}_wt"]  = wts
    miv_raw[f"{metric}_att"] = atts
    miv_raw[f"{metric}_sc"]  = scs
    miv_raw[f"{metric}_ws"]  = wss
    lob_score += np.array(wss)
    point     += np.array([(a * w if a is not None else 0.0) for a, w in zip(atts, wts)])

miv_raw["lob_score"]        = lob_score
miv_raw["point"]            = point
_total_chats                = pd.to_numeric(miv_raw["chat_count"], errors="coerce").fillna(0).sum()
miv_raw["final_score"]      = np.where(_total_chats > 0, miv_raw["lob_score"] * pd.to_numeric(miv_raw["chat_count"], errors="coerce").fillna(0) / _total_chats, 0)
miv_raw["key_unique_month"] = miv_raw["_PST.Month"].astype(str) + "_" + miv_raw["Agent Email ID"].astype(str)

miv_final = miv_raw.assign(**{
    "LC(%)":          pd.to_numeric(miv_raw["lc"],      errors="coerce") * 100,
    "NPS(%)":         pd.to_numeric(miv_raw["nps"],     errors="coerce") * 100,
    "Exceed_Chat(%)": pd.to_numeric(miv_raw["exc"],     errors="coerce") * 100,
    "FCR(%)":         pd.to_numeric(miv_raw["fcr"],     errors="coerce"),
    "LC_%Att":        pd.to_numeric(miv_raw["LC_att"],  errors="coerce") * 100,
    "NPS_%Att":       pd.to_numeric(miv_raw["NPS_att"], errors="coerce") * 100,
    "AHT_%Att":       pd.to_numeric(miv_raw["AHT_att"], errors="coerce") * 100,
    "Exceed_%Att":    pd.to_numeric(miv_raw["EXC_att"], errors="coerce") * 100,
    "FCR_%Att":       pd.to_numeric(miv_raw["FCR_att"], errors="coerce") * 100,
    "QA_att_month":   pd.to_numeric(miv_raw["AWF_att"], errors="coerce") * 100,
}).rename(columns={
    "chat_count":  "#Vol",           "aht":           "AHT",
    "NPS_sc":      "NPS_score",      "LC_sc":         "LC_score",
    "AHT_sc":      "AHT_score",      "EXC_tar":       "Exceed_tar",
    "EXC_sc":      "Exceed_score",   "FCR_sc":        "FCR_score",
    "lob_score":   "LOB_score",      "_prom":         "_Promoter",
    "_detr":       "_Detractor",     "_surv":         "_Survey",
    "_lc_sum":     "Long Chat",      "_fcr_p":        "Passed Sessions",
    "_fcr_f":      "Failed Sessions","_fcr_t":        "Total Sessions",
    "_exceed_sum": "Exceed Chat",
    "AWF_tar":     "QA_tar_month",   "AWF_sc":        "QA_sc_month",
})

_COL_ORDER = [
    "key_unique_month", "_PST.Month", "_PST.Date", "_VN.Date",
    "OracleID", "Employee Name", "Agent Email ID", "People ID",
    "Designation", "Detail Status", "Group", "Active", "TL ID",
    "Supervisor Email", "Supervisor Name", "SME", "LOB",
    "#Vol", "AHT", "LC(%)",
    "NPS_tar", "NPS_score", "LC_%Att", "LC_tar", "LC_score",
    "AHT_%Att", "AHT_tar", "AHT_score",
    "Exceed_Chat(%)", "Exceed_%Att", "Exceed_tar", "Exceed_score",
    "FCR_tar", "FCR_score", "NPS(%)", "NPS_%Att", "FCR(%)", "FCR_%Att",
    "QA_Score_month", "QA_tar_month", "QA_att_month", "QA_sc_month",
    "HC_Unplanned_month", "HC_Schedule_month", "Unplanned_pct_month",
    "ATT_Movement_month", "LWD_Movement_month",
    "LOB_score",
    "Passed Sessions", "Failed Sessions", "Total Sessions",
    "Duplicate_Flag", "_Promoter", "_Detractor", "_Neutral", "_Survey",
    "Exceed Chat", "Long Chat",
]

miv_final = miv_final[[c for c in _COL_ORDER if c in miv_final.columns]]

_out = folder_paths["output_miv_performance"]
os.makedirs(_out, exist_ok=True)
for month, grp in miv_final.groupby("_PST.Month"):
    pl.from_pandas(grp).write_parquet(os.path.join(_out, f"miv_agent_{month}.parquet"))

print(f"miv_final: {len(miv_final)} rows x {len(miv_final.columns)} cols")
miv_final.to_excel("test.xlsx")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_24300\4041302828.py:217: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  miv_raw = pd.concat([miv_raw, _missing], ignore_index=True, sort=False)


Added 206 filler rows (no volume agents)
miv_final: 623 rows x 58 cols


In [38]:
# Export per month + total parquet (diagonal_relaxed để merge với tháng cũ)
output_dir   = folder_paths["output_miv_performance"]
os.makedirs(output_dir, exist_ok=True)

for (month_value,), group in performance_hcm.group_by(['_PST.Month'], maintain_order=True):
    base_name = str(month_value)
    group.write_csv(os.path.join(output_dir, f"{base_name}.csv"))

parquet_file     = "_miv_performance_hcm.parquet"
out_path_parquet = os.path.join(output_dir, parquet_file)
performance_hcm.write_parquet(out_path_parquet)

print(f"Exported {performance_hcm.height:,} rows → {out_path_parquet}")

NameError: name 'performance_hcm' is not defined

In [ ]:
print(performance_all_site['_PST.Month'].drop_nulls().unique().sort())
print(performance_all_site.schema)

shape: (2,)
Series: '_PST.Month' [str]
[
	"26_06"
	"26_07"
]
Schema({'Agent People Id': String, 'Business Segment Name': String, 'Partner Name': String, 'Response Count': String, 'Response Time': String, 'Latest VA Product': String, 'Language': String, 'Latest VA Intent': String, 'Conversation Id': String, 'Agent Queue Group Name': String, 'Joined Time': Datetime(time_unit='us', time_zone=None), '_PST.Interval': String, 'Agent Email ID': String, 'Handle (Count)': Int64, 'Handle Time (Sum)': Float64, 'Hold Time (Sum)': Float64, 'Talk Time (Sum)': Float64, 'Join Time (VNT)': Datetime(time_unit='us', time_zone=None), '_VNT.Interval': String, '_VNT.Period': String, 'Wrap Up Time (Sum)': Float64, 'Agent Business Location': String, '_PST.Date': Date, '_PST.Month': String, '_PST.Year': Int32, '_aob': Int32, 'LOB': String, '_conver_unique': String, '_nps_type': String, '_promoter': Int32, '_detractor': Int32, '_neutral': Int32, '_survey': Int32, 'DUET': Int32, '_verbatim': String, 'T3': Int32,